In [1]:
from google_play_scraper import reviews, Sort
import pandas as pd

# 앱 패키지명
APP_ID = "com.ringle"

# 리뷰 수집
result, continuation_token = reviews(
    APP_ID,
    lang="ko",          # 한국어 리뷰 우선
    country="kr",       # 한국 스토어 기준
    sort=Sort.NEWEST,   # 최신순
    count=200           # 가져올 리뷰 수
)

# 데이터프레임으로 정리
df = pd.DataFrame(result)

# 필요한 컬럼만 선택
df = df[
    [
        "userName",
        "score",
        "at",
        "content",
        "thumbsUpCount",
        "reviewCreatedVersion"
    ]
]

# 컬럼명 한글로 변경
df.columns = [
    "작성자",
    "별점",
    "작성일",
    "리뷰내용",
    "좋아요수",
    "앱버전"
]

# 저장
df.to_csv("ringle_googleplay_reviews.csv", index=False, encoding="utf-8-sig")

print(df.head())
print(f"\n총 {len(df)}개 리뷰 저장 완료: ringle_googleplay_reviews.csv")

    작성자  별점                 작성일  \
0    HK   2 2026-03-08 00:23:46   
1    박학   4 2026-02-19 08:35:24   
2  백만볼트   4 2026-01-15 22:58:10   
3   파도하   1 2025-12-28 23:37:38   
4   SHI   3 2025-12-27 10:02:18   

                                                리뷰내용  좋아요수    앱버전  
0  서버 개발중이신지는 모르겠으나 AI모드가 너무...속도가느리고 끊깁니다ㅠㅠ 구성이나...     0   None  
1  지금까지 PC로 미국 튜터랑 1:1 수업 7년 동안 받아온 링글 사용자 입니다. 3...     0  9.7.0  
2  1대1 튜터링 매우 만족스럽게 쓰고 있습니다. 다만 교재 사전질문 선택시 다른 질문...     0  9.5.1  
3  갑자기 AI튜터 관련 메뉴들이 터치가 1도 안먹네요. 수업진행을 할수가없는데 기간은...     0  9.5.0  
4  1. 앱 업데이트후 AI 듣기에서 찜하기가 없어졌습니다 2. 안드로이드는 AI 듣기...     0  9.5.0  

총 168개 리뷰 저장 완료: ringle_googleplay_reviews.csv


In [3]:
import pandas as pd

df = pd.read_csv("ringle_googleplay_reviews.csv")

keywords = [
    "인식", "못 알아듣", "발음", "분석",
    "터치", "스크롤", "반복", "백그라운드",
    "끊김", "로그인", "버그", "오류"
]

def contains_keyword(text):
    text = str(text)
    return any(keyword in text for keyword in keywords)

filtered_df = df[df["리뷰내용"].apply(contains_keyword)].copy()

filtered_df.to_csv("ringle_reviews_filtered.csv", index=False, encoding="utf-8-sig")

print(filtered_df[["작성일", "별점", "리뷰내용"]].head(20))
print(f"\n키워드 관련 리뷰 수: {len(filtered_df)}개")

                     작성일  별점  \
2    2026-01-15 22:58:10   4   
3    2025-12-28 23:37:38   1   
4    2025-12-27 10:02:18   3   
5    2025-12-26 16:22:40   1   
7    2025-12-05 19:48:54   1   
8    2025-12-03 07:46:21   2   
27   2024-08-27 14:37:54   3   
57   2023-10-17 21:07:36   1   
63   2023-02-08 17:04:24   1   
66   2023-01-24 14:59:57   2   
71   2022-12-21 17:51:24   4   
79   2022-11-06 20:55:11   3   
91   2022-07-21 07:35:18   2   
92   2022-07-02 10:20:43   4   
94   2022-05-06 14:08:56   1   
97   2022-03-02 22:27:02   3   
108  2021-11-15 18:59:09   1   
117  2021-07-30 20:23:00   4   
130  2021-02-01 09:21:31   2   

                                                  리뷰내용  
2    1대1 튜터링 매우 만족스럽게 쓰고 있습니다. 다만 교재 사전질문 선택시 다른 질문...  
3    갑자기 AI튜터 관련 메뉴들이 터치가 1도 안먹네요. 수업진행을 할수가없는데 기간은...  
4    1. 앱 업데이트후 AI 듣기에서 찜하기가 없어졌습니다 2. 안드로이드는 AI 듣기...  
5                          오류나서 회원가입도 못하고 로그인도 못하고 있어요  
7                                           오류가 너무 많아요  
8    다른건 다 좋은데, 수

In [4]:
import pandas as pd

df = pd.read_csv("ringle_googleplay_reviews.csv")

categories = {
    "인식_분석_문제": ["인식", "못 알아듣", "발음", "분석"],
    "UX_조작성_문제": ["터치", "스크롤", "버튼", "화면", "불편"],
    "안정성_문제": ["끊김", "버그", "오류", "멈춤", "튕김"],
    "반복_복습_문제": ["반복", "복습", "백그라운드"],
    "접근_로그인_문제": ["로그인", "접속", "인증"]
}

def classify_review(text):
    text = str(text)
    matched = []
    for category, words in categories.items():
        if any(word in text for word in words):
            matched.append(category)
    return ", ".join(matched) if matched else "기타"

df["카테고리"] = df["리뷰내용"].apply(classify_review)

# 카테고리별 리뷰 수
category_counts = df["카테고리"].value_counts()
print(category_counts)

# 저장
df.to_csv("ringle_reviews_classified.csv", index=False, encoding="utf-8-sig")

카테고리
기타                      120
UX_조작성_문제                22
안정성_문제                    8
접근_로그인_문제                 6
UX_조작성_문제, 접근_로그인_문제      4
UX_조작성_문제, 안정성_문제         4
반복_복습_문제                  2
안정성_문제, 접근_로그인_문제         1
UX_조작성_문제, 반복_복습_문제       1
Name: count, dtype: int64


In [5]:
import pandas as pd

df = pd.read_csv("ringle_reviews_classified.csv")

# 너무 짧은 리뷰 제외
df["리뷰길이"] = df["리뷰내용"].astype(str).apply(len)
candidate_df = df[df["리뷰길이"] >= 15].copy()

# 별점 낮은 순, 최신순 정렬
candidate_df = candidate_df.sort_values(
    by=["별점", "작성일"],
    ascending=[True, False]
)

print(candidate_df[["작성일", "별점", "카테고리", "리뷰내용"]].head(20))

                    작성일  별점               카테고리  \
3   2025-12-28 23:37:38   1          UX_조작성_문제   
5   2025-12-26 16:22:40   1  안정성_문제, 접근_로그인_문제   
6   2025-12-17 19:04:54   1                 기타   
9   2025-10-20 20:38:59   1          UX_조작성_문제   
11  2025-08-03 23:47:22   1          접근_로그인_문제   
13  2025-06-20 00:57:26   1          UX_조작성_문제   
19  2024-10-25 23:21:59   1                 기타   
20  2024-10-22 23:07:11   1          접근_로그인_문제   
21  2024-10-22 01:43:08   1          UX_조작성_문제   
23  2024-10-18 20:33:30   1                 기타   
26  2024-08-29 13:09:47   1                 기타   
31  2024-07-28 20:02:11   1                 기타   
36  2024-05-16 20:21:14   1                 기타   
39  2024-04-29 18:08:50   1          접근_로그인_문제   
43  2024-03-28 10:05:39   1                 기타   
49  2024-01-06 18:19:11   1                 기타   
52  2023-12-08 16:17:55   1                 기타   
53  2023-12-05 21:14:46   1          UX_조작성_문제   
54  2023-11-07 23:19:33   1                 기타   
